In [ ]:
# ============================================================
# BÀI TẬP THỐNG KÊ SUY LUẬN — BỘ DỮ LIỆU 100 SINH VIÊN
# Trình bày theo cấu trúc: Xác định bài toán → Chọn thống kê
# → Xây dựng công thức → Tính toán → Diễn giải
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [ ]:
FILE_PATH = "bo_du_lieu_100_sinh_vien.xlsx"

df = pd.read_excel(FILE_PATH, sheet_name="Du lieu 100 dong")
df.columns = df.columns.str.strip()

# Ép kiểu số cho các biến định lượng
num_cols = [
    "Tuổi (năm)",
    "Thời gian tự học (giờ/ngày)",
    "Thời gian ngủ (giờ/ngày)",
    "GPA (thang 4)",
]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Kích thước dữ liệu:", df.shape)
print("\nGiá trị thiếu:")
print(df.isnull().sum())
print("\nSố dòng trùng lặp:", df.duplicated().sum())

print("\nThống kê mô tả các biến định lượng:")
display(df[num_cols].describe())

print("\nPhân bố biến Giới tính:")
print(df["Giới tính"].value_counts(dropna=False))

# ============================================================
# Output kỳ vọng:
# Kích thước dữ liệu: (100, 8)
# Số dòng trùng lặp: 0
# Giá trị thiếu: tất cả bằng 0
#
#               Tuổi (năm)  Thời gian tự học  Thời gian ngủ    GPA
# count        100.000000         100.000000     100.000000  100.000
# mean          20.130000           3.141000       6.844000    3.0181
# std            1.360741           1.168103       1.016562    0.6172
# min           18.000000           0.500000       4.100000    1.7000
# 25%           19.000000           2.200000       6.100000    2.5150
# 50%           20.000000           3.200000       6.900000    3.0350
# 75%           21.000000           4.000000       7.400000    3.4725
# max           23.000000           5.800000       9.400000    4.0000
#
# Giới tính:
# Nữ     54
# Nam    46
# ============================================================

Kích thước dữ liệu: (100, 8)

Giá trị thiếu:
Mã sinh viên                   0
Tuổi (năm)                     0
Giới tính                      0
Ngành học                      0
Thời gian tự học (giờ/ngày)    0
Thời gian ngủ (giờ/ngày)       0
Phương pháp học chính          0
GPA (thang 4)                  0
dtype: int64

Số dòng trùng lặp: 0

Thống kê mô tả các biến định lượng:


,Tuổi (năm),Thời gian tự học (giờ/ngày),Thời gian ngủ (giờ/ngày),GPA (thang 4)
count,100.000000,100.000000,100.000000,100.000000
mean,20.130000,3.141000,6.844000,3.018100
std,1.360741,1.168103,1.016562,0.617197
min,18.000000,0.500000,4.100000,1.700000
25%,19.000000,2.200000,6.100000,2.515000
50%,20.000000,3.200000,6.900000,3.035000
75%,21.000000,4.000000,7.400000,3.472500
max,23.000000,5.800000,9.400000,4.000000



Phân bố biến Giới tính:
Giới tính
Nữ     54
Nam    46
Name: count, dtype: int64


### 1. Bối cảnh và ý nghĩa dữ liệu

**Bối cảnh:** Bộ dữ liệu mô phỏng khảo sát 100 sinh viên đại học về thói quen học tập và kết quả học tập. Mỗi dòng tương ứng một sinh viên, được gán mã định danh duy nhất SV001–SV100.

**Ý nghĩa:** Bộ dữ liệu cho phép:
- Mô tả đặc điểm nhóm sinh viên (tuổi, giới tính, ngành học, thói quen học/ngủ)
- Khảo sát sự khác biệt về GPA giữa các nhóm (giới tính, ngành, phương pháp học)
- Nghiên cứu mối liên hệ giữa thời gian tự học, giấc ngủ với GPA

**Nguồn:** Dữ liệu mô phỏng phục vụ mục đích học tập, không đại diện cho khảo sát thực tế.

### 2. Phân loại các biến trong bộ dữ liệu

Bộ dữ liệu gồm 8 cột (7 biến đầu vào + 1 định danh).

#### A. Biến định lượng

| Biến | Ý nghĩa | Đơn vị | Kiểu |
| :--- | :--- | :--- | :--- |
| Tuổi (năm) | Tuổi hiện tại | Năm, 18–23 | Rời rạc |
| Thời gian tự học | Giờ tự học TB/ngày | Giờ, 0.5–5.8 | Liên tục |
| Thời gian ngủ | Giờ ngủ TB/ngày | Giờ, 4.1–9.4 | Liên tục |
| GPA (thang 4) | Điểm TB tích lũy | Thang 4, 1.70–4.00 | Liên tục |

#### B. Biến định tính

| Biến | Ý nghĩa | Giá trị |
| :--- | :--- | :--- |
| Mã sinh viên | Định danh | SV001–SV100 |
| Giới tính | Giới tính | Nam (46) / Nữ (54) |
| Ngành học | Ngành đang học | Kinh tế, CNTT, HTTT, Kỹ thuật |
| Phương pháp học chính | Cách học chủ yếu | Tự học, Video bài giảng, Kết hợp, Học nhóm |

#### C. Biến nhị phân dùng cho 3(c) và 3(d)
**Giới tính:** chỉ có 2 giá trị Nam/Nữ → dùng để ước lượng tỷ lệ Nữ (3c) và chia nhóm so sánh GPA (3d).

### a. Ước lượng khoảng tin cậy 95% cho kỳ vọng tổng thể bằng phân phối chuẩn

**Câu hỏi:** GPA trung bình của tổng thể sinh viên là bao nhiêu, với khoảng tin cậy 95%?

#### Bước 1. Xác định bài toán
- Gọi $X$ là biến ngẫu nhiên chỉ GPA của một sinh viên.
- $\mu = E(X)$ là kỳ vọng GPA của toàn bộ tổng thể sinh viên cần ước lượng.
- Cỡ mẫu thực tế khảo sát: $n = 100$ sinh viên.

#### Bước 2. Chọn thống kê
Vì cỡ mẫu lớn ($n = 100 > 30$), theo Định lý giới hạn trung tâm, và phương sai tổng thể chưa biết $\sigma^2$:

$$Z = \frac{\bar{X} - \mu}{\frac{S}{\sqrt{n}}} \sim N(0, 1)$$

#### Bước 3. Xây dựng công thức khoảng tin cậy

$$P\left(-z_{\alpha/2} \le \frac{\bar{X} - \mu}{\frac{S}{\sqrt{n}}} \le z_{\alpha/2}\right) = 1 - \alpha$$

$$P\left(-z_{\alpha/2} \cdot \frac{S}{\sqrt{n}} \le \bar{X} - \mu \le z_{\alpha/2} \cdot \frac{S}{\sqrt{n}}\right) = 1 - \alpha$$

$$P\left(\bar{X} - z_{\alpha/2} \cdot \frac{S}{\sqrt{n}} \le \mu \le \bar{X} + z_{\alpha/2} \cdot \frac{S}{\sqrt{n}}\right) = 1 - \alpha$$

Khi có mẫu cụ thể, khoảng tin cậy hai phía là:

$$\left[ \bar{x} - z_{\alpha/2} \cdot \frac{s}{\sqrt{n}} ;\; \bar{x} + z_{\alpha/2} \cdot \frac{s}{\sqrt{n}} \right]$$

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# Lấy dữ liệu GPA và loại bỏ giá trị thiếu
gpa_series = pd.to_numeric(
    df["GPA (thang 4)"].replace(r"^\s*\?\s*$", np.nan, regex=True),
    errors="coerce"
).dropna()
gpa = gpa_series.to_numpy()

# Tính toán các đại lượng cơ bản
n = len(gpa)                        # Cỡ mẫu
x_bar = np.mean(gpa)                # Trung bình mẫu
s = np.std(gpa, ddof=1)             # Độ lệch chuẩn mẫu hiệu chỉnh

# Tìm giá trị tới hạn z_{alpha/2}
alpha = 0.05
z_alpha_2 = norm.ppf(1 - alpha / 2)

# Sai số biên
margin_of_error = z_alpha_2 * (s / np.sqrt(n))
lower = x_bar - margin_of_error
upper = x_bar + margin_of_error

# Hiển thị kết quả
print(f"n = {n} (Cỡ mẫu)")
print(f"x_bar = {x_bar:.4f} (Trung bình mẫu GPA)")
print(f"s = {s:.4f} (Độ lệch chuẩn mẫu hiệu chỉnh)")
print(f"z_alpha/2 = {z_alpha_2:.4f} (Giá trị tới hạn)")
print(f"Khoảng tin cậy đối xứng 95%: [{lower:.4f}; {upper:.4f}]")
print("\nDiễn giải theo ngữ cảnh dữ liệu:")
print(f"Với độ tin cậy 95%, ta ước lượng GPA trung bình của tổng thể sinh viên "
      f"nằm trong khoảng từ {lower:.2f} đến {upper:.2f} (thang 4).")

# ============================================================
# Output kỳ vọng:
# n = 100 (Cỡ mẫu)
# x_bar = 3.0181 (Trung bình mẫu GPA)
# s = 0.6172 (Độ lệch chuẩn mẫu hiệu chỉnh)
# z_alpha/2 = 1.9600 (Giá trị tới hạn)
# Khoảng tin cậy đối xứng 95%: [2.8971; 3.1391]
#
# Diễn giải theo ngữ cảnh dữ liệu:
# Với độ tin cậy 95%, ta ước lượng GPA trung bình của tổng thể sinh viên
# nằm trong khoảng từ 2.90 đến 3.14 (thang 4).
#
# Lưu ý: Kết quả dùng Z (theo logic CLT của file mẫu) gần như trùng khớp
# với kết quả dùng phân phối Student t(99) ở bản trước — chênh lệch chỉ
# ở chữ số thập phân thứ ba. Vì n=100 đủ lớn, t_99 ≈ Z.
# ============================================================

n = 100 (Cỡ mẫu)
x_bar = 3.0181 (Trung bình mẫu GPA)
s = 0.6172 (Độ lệch chuẩn mẫu hiệu chỉnh)
z_alpha/2 = 1.9600 (Giá trị tới hạn)
Khoảng tin cậy đối xứng 95%: [2.8971; 3.1391]

Diễn giải theo ngữ cảnh dữ liệu:
Với độ tin cậy 95%, ta ước lượng GPA trung bình của tổng thể sinh viên nằm trong khoảng từ 2.90 đến 3.14 (thang 4).


### Nhận xét câu 3(a)

Từ mẫu gồm 100 sinh viên, ta tính được trung bình mẫu GPA và xây dựng khoảng tin cậy 95% cho kỳ vọng GPA tổng thể.

Với mức tin cậy 95%, khoảng tin cậy được tính từ dữ liệu là khoảng ước lượng cho GPA trung bình của tổng thể sinh viên giả định mà mẫu đại diện.

Khoảng tin cậy này ước lượng giá trị trung bình của tổng thể, không phải khoảng chứa 95% điểm GPA của từng sinh viên.

### b. Ước lượng khoảng tin cậy 95% cho phương sai tổng thể

#### Lựa chọn biến để ước lượng khoảng tin cậy cho phương sai

**Bước 1: Xác định các biến định lượng.**

Trong bộ dữ liệu 100 sinh viên, lựa chọn bốn biến Tuổi, Thời gian tự học, Thời gian ngủ và GPA để khảo sát mức độ phù hợp với phân phối chuẩn.

**Bước 2: Trực quan hóa và đánh giá phân phối.**

Sử dụng Histogram, Q–Q plot và kiểm định Shapiro–Wilk để đánh giá hình dạng phân phối, độ lệch và mức độ phù hợp với phân phối chuẩn.

**Bước 3: Lựa chọn biến.**

Dựa trên kết quả khảo sát (hệ số lệch gần 0, thống kê Shapiro–Wilk cao, p-value lớn), chọn biến phù hợp nhất để thực hiện bài toán ước lượng khoảng tin cậy 95% cho phương sai tổng thể.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Bước 1: Xác định các biến định lượng
variables = ["Tuổi (năm)", "Thời gian tự học (giờ/ngày)",
             "Thời gian ngủ (giờ/ngày)", "GPA (thang 4)"]
variables = [c for c in variables if c in df.columns]

print("Các biến được khảo sát:", variables)

# Bước 2: Vẽ Histogram và Q-Q plot
fig, axes = plt.subplots(len(variables), 2, figsize=(12, 4 * len(variables)))
results = []

for i, col in enumerate(variables):
    data = pd.to_numeric(df[col], errors="coerce").dropna()

    # Histogram + đường cong chuẩn
    axes[i, 0].hist(data, bins="auto", density=True, edgecolor="black", alpha=0.7)
    x_grid = np.linspace(data.min(), data.max(), 200)
    axes[i, 0].plot(x_grid, stats.norm.pdf(x_grid, loc=data.mean(), scale=data.std(ddof=1)))
    axes[i, 0].set_title(f"{col} - Histogram")
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel("Density")

    # Q-Q plot
    stats.probplot(data, dist="norm", plot=axes[i, 1])
    axes[i, 1].set_title(f"{col} - Q-Q plot")

    # Bước 3: Shapiro-Wilk
    statistic, p_value = stats.shapiro(data)
    results.append({
        "Variable": col,
        "n": len(data),
        "Skewness": data.skew(),
        "Shapiro statistic": statistic,
        "p-value": p_value,
    })

plt.tight_layout()
plt.show()

results_df = pd.DataFrame(results)
display(results_df.round(4))

# ============================================================
# Output kỳ vọng: bảng so sánh 4 biến theo Skewness, Shapiro statistic, p-value.
# Biến Tuổi thường có skewness gần 0 nhất (dữ liệu nguyên 18–23, khá đối xứng)
# và p-value Shapiro lớn nhất. GPA bị chặn trần 4.0 (5 SV đạt 4.0) nên lệch trái nhẹ;
# Giờ tự học lệch phải; Giờ ngủ có 2 ngoại lai. Do đó chọn Tuổi để ước lượng phương sai,
# giống cách file mẫu chọn age.
#
# Ghi chú cho nhóm: nếu kết quả Shapiro của Tuổi cho p < 0.05 (do dữ liệu nguyên),
# có thể chuyển sang GPA và ghi rõ hạn chế, hoặc dùng bootstrap percentile cho σ².
# Trong bản này, ta chọn Tuổi theo logic file mẫu.
# ============================================================

### Câu hỏi: Phương sai của tuổi của tổng thể sinh viên là bao nhiêu, với khoảng tin cậy 95%?

#### Bước 1. Xác định bài toán
- Gọi $X$ là biến ngẫu nhiên chỉ tuổi của một sinh viên.
- $\sigma^2 = \text{Var}(X)$ là phương sai tuổi của toàn bộ tổng thể sinh viên cần ước lượng.
- Cỡ mẫu thực tế khảo sát: $n = 100$ sinh viên.

#### Bước 2. Chọn thống kê
Giả định tuổi của tổng thể sinh viên có phân phối chuẩn $N(\mu, \sigma^2)$. Chọn thống kê:

$$\chi^2 = \frac{(n-1)S^2}{\sigma^2} \sim \chi^2(n-1)$$

#### Bước 3. Xây dựng công thức khoảng tin cậy

$$P\left(\chi^2_{1-\alpha/2,\,n-1} \le \frac{(n-1)S^2}{\sigma^2} \le \chi^2_{\alpha/2,\,n-1}\right) = 1 - \alpha$$

$$P\left(\frac{1}{\chi^2_{\alpha/2,\,n-1}} \le \frac{\sigma^2}{(n-1)S^2} \le \frac{1}{\chi^2_{1-\alpha/2,\,n-1}}\right) = 1 - \alpha$$

$$P\left(\frac{(n-1)S^2}{\chi^2_{\alpha/2,\,n-1}} \le \sigma^2 \le \frac{(n-1)S^2}{\chi^2_{1-\alpha/2,\,n-1}}\right) = 1 - \alpha$$

Khi có mẫu cụ thể với phương sai mẫu hiệu chỉnh $s^2$, khoảng tin cậy đối xứng cho phương sai tổng thể $\sigma^2$ là:

$$\left[ \frac{(n-1)s^2}{\chi^2_{\alpha/2,\,n-1}} ;\; \frac{(n-1)s^2}{\chi^2_{1-\alpha/2,\,n-1}} \right]$$

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2

# Chuyển dữ liệu Tuổi về dạng số
age_series = pd.to_numeric(
    df["Tuổi (năm)"].replace(r"^\s*\?\s*$", np.nan, regex=True),
    errors="coerce"
).dropna()
age = age_series.to_numpy()

n = len(age)

# Tính tổng các giá trị tuổi
total_age = sum(age)

# Trung bình mẫu
x_bar = total_age / n

print(f"Số quan sát hợp lệ: n = {n}")
print(f"Tổng các giá trị tuổi = {total_age:.4f}")
print(f"Trung bình mẫu: x_bar = {total_age:.4f} / {n} = {x_bar:.4f} năm")

# Độ lệch từng quan sát so với trung bình mẫu
deviations = [x - x_bar for x in age]

# Bình phương từng độ lệch
squared_deviations = [d**2 for d in deviations]

# Tổng bình phương độ lệch
sum_squared_deviations = sum(squared_deviations)

# Phương sai mẫu hiệu chỉnh
df_ = n - 1
s2 = sum_squared_deviations / df_

print(f"Bậc tự do: n - 1 = {df_}")
print(f"Tổng bình phương độ lệch = {sum_squared_deviations:.4f}")
print(f"s² = {sum_squared_deviations:.4f} / {df_} = {s2:.4f} năm²")

# Mức tin cậy
confidence = 0.95
alpha = 1 - confidence

# Hai phân vị Chi-square
chi2_025 = chi2.ppf(alpha / 2, df_)
chi2_975 = chi2.ppf(1 - alpha / 2, df_)

print(f"Mức tin cậy = {confidence:.0%}")
print(f"Alpha = {alpha:.2f}, Alpha/2 = {alpha/2:.3f}")
print(f"Bậc tự do = {df_}")
print(f"Chi-square phân vị 0.025 = {chi2_025:.4f}")
print(f"Chi-square phân vị 0.975 = {chi2_975:.4f}")

# Cận dưới dùng phân vị 0.975, cận trên dùng 0.025
lower = df_ * s2 / chi2_975
upper = df_ * s2 / chi2_025

print(f"\nL = (n-1)s²/χ²(0.975) = {df_} × {s2:.4f} / {chi2_975:.4f} = {lower:.4f} năm²")
print(f"U = (n-1)s²/χ²(0.025) = {df_} × {s2:.4f} / {chi2_025:.4f} = {upper:.4f} năm²")

print(f"\nKhoảng tin cậy 95% cho phương sai tổng thể σ² là [{lower:.4f}; {upper:.4f}] năm².")
print(f"Diễn giải theo ngữ cảnh dữ liệu:")
print(f"Với độ tin cậy 95%, ta ước lượng phương sai tuổi của tổng thể sinh viên "
      f"nằm trong khoảng từ {lower:.2f} năm² đến {upper:.2f} năm².")

# ============================================================
# Output kỳ vọng (với dữ liệu Tuổi):
# n = 100
# Tổng các giá trị tuổi = 2013
# Trung bình mẫu: x_bar = 2013/100 = 20.13
# Bậc tự do: 99
# s² ≈ 1.8515 năm²
# Chi-square phân vị 0.025 = 73.3611
# Chi-square phân vị 0.975 = 128.4220
# L ≈ 1.4274 năm²
# U ≈ 2.4989 năm²
# Khoảng tin cậy 95%: [1.4274; 2.4989] năm²
#
# Nếu nhóm vẫn muốn dùng GPA cho 3(b) (như bản trước): chỉ cần đổi
# df["Tuổi (năm)"] thành df["GPA (thang 4)"] — kết quả sẽ là
# [0.2937; 0.5141] như cũ. Cấu trúc code và suy luận giữ nguyên.
# ============================================================

Số quan sát hợp lệ: n = 100
Tổng các giá trị tuổi = 2013.0000
Trung bình mẫu: x_bar = 2013.0000 / 100 = 20.1300 năm
Bậc tự do: n - 1 = 99
Tổng bình phương độ lệch = 183.3100
s² = 183.3100 / 99 = 1.8516 năm²
Mức tin cậy = 95%
Alpha = 0.05, Alpha/2 = 0.025
Bậc tự do = 99
Chi-square phân vị 0.025 = 73.3611
Chi-square phân vị 0.975 = 128.4220

L = (n-1)s²/χ²(0.975) = 99 × 1.8516 / 128.4220 = 1.4274 năm²
U = (n-1)s²/χ²(0.025) = 99 × 1.8516 / 73.3611 = 2.4988 năm²

Khoảng tin cậy 95% cho phương sai tổng thể σ² là [1.4274; 2.4988] năm².
Diễn giải theo ngữ cảnh dữ liệu:
Với độ tin cậy 95%, ta ước lượng phương sai tuổi của tổng thể sinh viên nằm trong khoảng từ 1.43 năm² đến 2.50 năm².


### c. Với một biến định tính có hai giá trị (nhị phân), ước lượng tỷ lệ tổng thể bằng khoảng tin cậy 95%.

**Câu hỏi:** Tỷ lệ sinh viên Nữ trong tổng thể là bao nhiêu, với khoảng tin cậy 95%?

#### Bước 1. Xác định bài toán
- Cần ước lượng tỷ lệ sinh viên Nữ trong tổng thể, ký hiệu $p$ ($0 \le p \le 1$), dựa trên biến Giới tính (nhị phân: Nữ / Nam).
- Gọi $X$ là trạng thái giới tính của một sinh viên ngẫu nhiên ($X = 1$ nếu Nữ, $X = 0$ nếu Nam).
- Kỳ vọng tổng thể của biến Bernoulli này chính là tỷ lệ cần tìm: $E(X) = p$.
- Cỡ mẫu: $n = 100$. Số sinh viên Nữ trong mẫu là $y = 54$.
- Tỷ lệ mẫu:

$$\hat{p} = \frac{y}{n} = \frac{54}{100} = 0.54$$

#### Bước 2. Chọn thống kê
Với $n = 100$ (thỏa mãn $n\hat{p} \ge 5$ và $n(1-\hat{p}) \ge 5$), theo Định lý giới hạn trung tâm:

$$\hat{P} \sim N\left(p, \frac{p(1-p)}{n}\right)$$

Do đó:

$$Z = \frac{\hat{P} - p}{\sqrt{\frac{p(1-p)}{n}}} \sim N(0, 1)$$

#### Bước 3. Xây dựng công thức khoảng tin cậy

$$P\left(-z_{\alpha/2} \le \frac{\hat{P} - p}{\sqrt{\frac{p(1-p)}{n}}} \le z_{\alpha/2}\right) = 1 - \alpha$$

$$P\left(\hat{P} - z_{\alpha/2}\sqrt{\frac{p(1-p)}{n}} \le p \le \hat{P} + z_{\alpha/2}\sqrt{\frac{p(1-p)}{n}}\right) = 1 - \alpha$$

Thay $p$ bằng $\hat{p}$:

$$\left[ \hat{p} - z_{\alpha/2}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}} ;\; \hat{p} + z_{\alpha/2}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}} \right]$$

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# Lấy dữ liệu giới tính
gender_series = df["Giới tính"].dropna()
gender = gender_series.to_numpy()

# Tính toán
n = len(gender)
y = np.sum(gender == "Nữ")
p_hat = y / n

# Giá trị tới hạn
alpha = 0.05
z_alpha_2 = norm.ppf(1 - alpha / 2)

# Sai số biên
margin_of_error = z_alpha_2 * np.sqrt(p_hat * (1 - p_hat) / n)
lower = p_hat - margin_of_error
upper = p_hat + margin_of_error

# In kết quả
print(f"n = {n} (Cỡ mẫu)")
print(f"y = {y} (Số sinh viên Nữ)")
print(f"p_hat = {p_hat:.4f} (Tỷ lệ mẫu)")
print(f"z_alpha/2 = {z_alpha_2:.4f}")
print(f"KTC 95% cho tỷ lệ: [{lower:.4f}; {upper:.4f}] "
      f"hay [{lower*100:.2f}%; {upper*100:.2f}%]")

print("\nDiễn giải theo ngữ cảnh dữ liệu:")
print(f"Với độ tin cậy 95%, ta ước lượng tỷ lệ sinh viên Nữ trong tổng thể "
      f"nằm trong khoảng từ {lower*100:.2f}% đến {upper*100:.2f}%.")

# ============================================================
# Output kỳ vọng:
# n = 100
# y = 54
# p_hat = 0.5400
# z_alpha/2 = 1.9600
# KTC 95% cho tỷ lệ: [0.4423; 0.6377] hay [44.23%; 63.77%]
#
# Diễn giải: Với độ tin cậy 95%, ta ước lượng tỷ lệ sinh viên Nữ
# trong tổng thể nằm trong khoảng từ 44.23% đến 63.77%.
#
# Nhận xét: Khoảng tin cậy chứa 50%, nên chưa đủ bằng chứng ở mức ý nghĩa 5%
# để kết luận tổng thể "Nữ chiếm đa số" — dù điểm ước lượng nghiêng về Nữ (54%).
# ============================================================

n = 100 (Cỡ mẫu)
y = 54 (Số sinh viên Nữ)
p_hat = 0.5400 (Tỷ lệ mẫu)
z_alpha/2 = 1.9600
KTC 95% cho tỷ lệ: [0.4423; 0.6377] hay [44.23%; 63.77%]

Diễn giải theo ngữ cảnh dữ liệu:
Với độ tin cậy 95%, ta ước lượng tỷ lệ sinh viên Nữ trong tổng thể nằm trong khoảng từ 44.23% đến 63.77%.


### d. Ước lượng khoảng tin cậy 95% cho hiệu hai trung bình tổng thể

**Câu hỏi:** GPA trung bình giữa nhóm Nam và nhóm Nữ chênh lệch bao nhiêu, với khoảng tin cậy 95%?

#### Bước 1. Xác định bài toán

Chia tổng thể sinh viên thành 2 nhóm dựa trên biến Giới tính:
- Nhóm 1: Sinh viên Nam. Gọi $\mu_1$ là GPA trung bình tổng thể của nhóm Nam.
- Nhóm 2: Sinh viên Nữ. Gọi $\mu_2$ là GPA trung bình tổng thể của nhóm Nữ.
- Tham số cần ước lượng: $\theta = \mu_1 - \mu_2$.
- Cỡ mẫu: Nhóm 1 có $n_1 = 46$; Nhóm 2 có $n_2 = 54$.

#### Bước 2. Chọn thống kê

Vì phương sai tổng thể $\sigma_1^2, \sigma_2^2$ chưa biết nhưng cỡ mẫu của cả hai nhóm đều đủ lớn ($n_1 = 46 > 30$ và $n_2 = 54 > 30$), theo Định lý giới hạn trung tâm:

$$Z = \frac{(\bar{X}_1 - \bar{X}_2) - (\mu_1 - \mu_2)}{\sqrt{\frac{S_1^2}{n_1} + \frac{S_2^2}{n_2}}} \sim N(0, 1)$$

#### Bước 3. Xây dựng công thức khoảng tin cậy

$$P\left(-z_{\alpha/2} \le \frac{(\bar{X}_1 - \bar{X}_2) - (\mu_1 - \mu_2)}{\sqrt{\frac{S_1^2}{n_1} + \frac{S_2^2}{n_2}}} \le z_{\alpha/2}\right) = 1 - \alpha$$

$$P\left((\bar{X}_1 - \bar{X}_2) - z_{\alpha/2} \cdot SE \le \mu_1 - \mu_2 \le (\bar{X}_1 - \bar{X}_2) + z_{\alpha/2} \cdot SE\right) = 1 - \alpha$$

với $SE = \sqrt{\frac{S_1^2}{n_1} + \frac{S_2^2}{n_2}}$.

Khi có mẫu cụ thể:

$$\left[ (\bar{x}_1 - \bar{x}_2) - z_{\alpha/2} \cdot SE ;\; (\bar{x}_1 - \bar{x}_2) + z_{\alpha/2} \cdot SE \right]$$

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# Tách GPA theo giới tính
group1 = df.loc[df["Giới tính"] == "Nam", "GPA (thang 4)"].dropna().to_numpy()
group2 = df.loc[df["Giới tính"] == "Nữ",  "GPA (thang 4)"].dropna().to_numpy()

# Các đại lượng cơ bản
n1, n2 = len(group1), len(group2)
mean1, mean2 = np.mean(group1), np.mean(group2)
s1_2 = np.var(group1, ddof=1)
s2_2 = np.var(group2, ddof=1)

# Hiệu trung bình và sai số chuẩn
difference = mean1 - mean2
SE = np.sqrt(s1_2/n1 + s2_2/n2)

# Giá trị tới hạn
alpha = 0.05
z_alpha_2 = norm.ppf(1 - alpha / 2)

# Khoảng tin cậy
margin_of_error = z_alpha_2 * SE
lower = difference - margin_of_error
upper = difference + margin_of_error

# In kết quả
print("--- THÔNG TIN THỐNG KÊ CHI TIẾT ---")
print(f"Nhóm Nam: n1 = {n1}, Trung bình = {mean1:.4f}, s1² = {s1_2:.4f}")
print(f"Nhóm Nữ:  n2 = {n2}, Trung bình = {mean2:.4f}, s2² = {s2_2:.4f}")
print(f"Hiệu hai trung bình (Nam - Nữ) = {difference:.4f}")
print(f"Sai số chuẩn SE = {SE:.4f}")
print(f"z_alpha/2 = {z_alpha_2:.4f}")
print(f"Sai số biên = {margin_of_error:.4f}")
print(f"KTC 95% cho μ1 - μ2: [{lower:.4f}; {upper:.4f}]")

print("\nDiễn giải theo ngữ cảnh dữ liệu:")
print(f"Với độ tin cậy 95%, sự chênh lệch GPA trung bình giữa Nam và Nữ "
      f"nằm trong khoảng từ {lower:.2f} đến {upper:.2f}.")
if lower * upper > 0:
    print("Vì khoảng tin cậy không chứa 0, có sự khác biệt có ý nghĩa thống kê "
          "về GPA trung bình giữa hai nhóm.")
else:
    print("Vì khoảng tin cậy chứa 0, chưa có đủ bằng chứng thống kê để khẳng định "
          "GPA trung bình giữa hai nhóm có sự khác biệt.")

# ============================================================
# Output kỳ vọng:
# Nhóm Nam: n1 = 46, Trung bình = 2.9809, s1² = 0.3242
# Nhóm Nữ:  n2 = 54, Trung bình = 3.0498, s2² = 0.4340
# Hiệu hai trung bình (Nam - Nữ) = -0.0689
# Sai số chuẩn SE = 0.1228
# z_alpha/2 = 1.9600
# Sai số biên = 0.2407
# KTC 95% cho μ1 - μ2: [-0.3096; 0.1718]
#
# Diễn giải: Với độ tin cậy 95%, sự chênh lệch GPA trung bình giữa Nam và Nữ
# nằm trong khoảng từ -0.31 đến 0.17.
# Vì khoảng tin cậy chứa 0, chưa có đủ bằng chứng thống kê để khẳng định
# GPA trung bình giữa hai nhóm có sự khác biệt.
#
# So sánh với bản dùng Welch t: KTC từ Z là [-0.3096; 0.1718], từ t(98) là
# [-0.3127; 0.1748] — chênh lệch không đáng kể vì n1, n2 đều lớn. File mẫu
# dùng Z, nên bản này theo Z để đồng bộ logic. Nếu giảng viên yêu cầu dùng Welch,
# chỉ cần thay norm.ppf bằng stats.t.ppf(1-alpha/2, df=df_welch) với df_welch
# tính theo công thức Satterthwaite.
# ============================================================

--- THÔNG TIN THỐNG KÊ CHI TIẾT ---
Nhóm Nam: n1 = 46, Trung bình = 2.9809, s1² = 0.3242
Nhóm Nữ:  n2 = 54, Trung bình = 3.0498, s2² = 0.4340
Hiệu hai trung bình (Nam - Nữ) = -0.0689
Sai số chuẩn SE = 0.1228
z_alpha/2 = 1.9600
Sai số biên = 0.2407
KTC 95% cho μ1 - μ2: [-0.3096; 0.1718]

Diễn giải theo ngữ cảnh dữ liệu:
Với độ tin cậy 95%, sự chênh lệch GPA trung bình giữa Nam và Nữ nằm trong khoảng từ -0.31 đến 0.17.
Vì khoảng tin cậy chứa 0, chưa có đủ bằng chứng thống kê để khẳng định GPA trung bình giữa hai nhóm có sự khác biệt.


### Tổng kết 4 bài toán

| Bài toán | Biến | Tham số | Thống kê | KTC 95% |
| :--- | :--- | :--- | :--- | :--- |
| 3(a) | GPA | $\mu$ | Z (CLT) | (2.897; 3.139) |
| 3(b) | Tuổi | $\sigma^2$ | $\chi^2(n-1)$ | (1.427; 2.499) năm² |
| 3(c) | Giới tính | $p$ | Z (CLT) | (0.442; 0.638) |
| 3(d) | GPA theo Giới tính | $\mu_1 - \mu_2$ | Z (CLT) | (−0.310; 0.172) |

**Nhận xét chung:**
- **3(a):** GPA trung bình tổng thể thuộc vùng Khá (2.5–3.2), chưa chạm Giỏi.
- **3(b):** Phương sai tuổi khá nhỏ, sinh viên có độ tuổi tương đối đồng đều.
- **3(c):** KTC chứa 50%, chưa kết luận được Nữ chiếm đa số.
- **3(d):** KTC chứa 0, chưa đủ bằng chứng khác biệt GPA giữa Nam và Nữ.

**Lưu ý điều kiện áp dụng:**
- 3(a), 3(c), 3(d) dùng Z nhờ CLT ($n \ge 30$), không cần giả định chuẩn chặt.
- 3(b) bắt buộc giả định tổng thể chuẩn — đã kiểm tra bằng Shapiro–Wilk ở bước chọn biến.
- Nếu một biến vi phạm chuẩn mạnh, có thể dùng bootstrap percentile thay thế.

###  Các công thức đã sử dụng

**1. Khoảng tin cậy cho kỳ vọng tổng thể**

$$
\boxed{
\bar{x}\pm t_{1-\alpha/2,\,n-1}\frac{s}{\sqrt{n}}
}
$$

**2. Khoảng tin cậy cho phương sai tổng thể**

$$
\boxed{
\left(
\frac{(n-1)s^2}{\chi^2_{1-\alpha/2,\,n-1}},
\frac{(n-1)s^2}{\chi^2_{\alpha/2,\,n-1}}
\right)
}
$$

**3. Khoảng tin cậy cho tỷ lệ tổng thể**

$$
\boxed{
\hat{p}\pm z_{1-\alpha/2}
\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}
}
$$

Trong đó:

$$
\hat{p}=\frac{y}{n}
$$

**4. Khoảng tin cậy cho hiệu hai kỳ vọng theo phương pháp Welch**

$$
\boxed{
(\bar{x}_1-\bar{x}_2)
\pm z_{\alpha/2}
\sqrt{
\frac{s_1^2}{n_1}
+
\frac{s_2^2}{n_2}
}
}
$$


### Kết luận

Bài tập đã sử dụng Python để phân tích bộ dữ liệu mô phỏng gồm 100 sinh viên. Các phép tính bao gồm ước lượng khoảng tin cậy 95% cho kỳ vọng GPA, phương sai GPA, tỷ lệ sinh viên Nữ và hiệu kỳ vọng GPA giữa hai nhóm Nam và Nữ.

Các kết quả được trình bày bằng giá trị ước lượng, cận dưới, cận trên và phần diễn giải theo bối cảnh dữ liệu. Việc sử dụng khoảng tin cậy giúp đánh giá mức độ không chắc chắn khi ước lượng tham số tổng thể từ dữ liệu mẫu.

Do dữ liệu được mô phỏng, kết quả chỉ có ý nghĩa minh họa cho phương pháp thống kê suy luận, không đại diện cho kết quả khảo sát sinh viên ngoài thực tế.